# 04a: Barley seeds, preprocessing and features

Paper section: Analysis of Real-World Barley Seed Data, the Experimental setup paragraph. This notebook is the reimplemented pipeline of Amézquita et al. (2022). Note that notebook 04b uses these features.

We run the following pipeline, per seed (3,121 seeds):

1. Clean the binary CT scan as in demeter
2. Reproduce Amézquita et al. pose and ECT: the principl component frame of the voxel coordinates with demeter's sign conventions, and the cubical ECT with 158 directions
3. Mesh the cleaned image with marching cubes at level 0.5, full resolution, in voxel units, centered at the voxel centroid. Then compute a mesh as a unit-weight simplicial complex.
4. Features are stored in `data/barley/features.npz`, and include:
   * the DIWECT of the mesh
   * pyECT's WECT with the paper's 158 directions and 16 heights on the same grid, with the seed in 10 poses
5. PCA flip-flop poses are stored in `data/barley/flipflop.npz`
6. Amézquita et al.'s binning are stored in `data/barley/binning.npz`

This notebook requires the extracted CT scans (`00-data-barley.ipynb`).

In [ ]:
import os
import time

import numpy as np
import pandas as pd
import torch
from joblib import Parallel, delayed
from pyect import WECT

import barley as B
from diwect import DIWECT

RECOMPUTE = False          # True: recompute all caches from the CT scans
VERIFY = 12                # seeds recomputed and compared with the cache (0 to skip)
N_JOBS = max(1, (os.cpu_count() or 2) - 2)

NUM_HEIGHTS_DIWECT = 64
PERTURBATIONS = [2, 5, 10, 20, 45, 90]   # degrees added to the paper's frame
POSES = ["paper", "automatic PCA", "scanner", "random rotation"] + [f"paper + {t}°" for t in PERTURBATIONS]
FLIP_POSES = ["paper", "sign flips", "width/thickness swap", "swap + sign flips"]
BIN_VARIANTS = [(cx, b, p) for cx in ["voxel", "mesh"] for b in ["min-max", "global"]
                for p in ["paper", "random rotation"]]
CACHE = {"frames": B.CACHE / "paper_frames.npz", "features": B.CACHE / "features.npz", "flipflop": B.CACHE / "flipflop.npz",
         "binning": B.CACHE / "binning.npz"}

d = B.load_table(with_files=True)
files, published_ect64 = d["files"], d["published_ect64"]
check = np.random.default_rng(B.SEED).choice(len(files), 20, replace=False)
print(f"{len(files)} seeds")

## Reproduce the paper's pose and ECT

demeter's pose is the SVD frame of the voxel coordinates. Since the SVD fixes each axis only up to sign, and demeter's rules fix only the third axis, the signs of the first two axes depend on the LAPACK library numpy is linked against (e.g., Accelerate on macOS and OpenBLAS on Linux). For each seed, we therefore keep the one of the four sign choices whose ECT (158 directions × 64 thresholds) reproduces the published ECT (or the closest one in $L_1$), so the pose is the same on every machine.


In [ ]:
def match_frame(path, published):
    coords, cells = B.cubical_complex(B.load_seed(path))
    _, rot, dist = B.matched_paper_frame(coords, cells, published)
    return rot, dist


B.CACHE.mkdir(parents=True, exist_ok=True)
if RECOMPUTE or not CACHE["frames"].exists():
    t0 = time.time()
    out = Parallel(n_jobs=N_JOBS)(delayed(match_frame)(files[i], published_ect64[i]) for i in range(len(files)))
    np.savez_compressed(CACHE["frames"], rot=np.stack([r for r, _ in out]), l1=np.array([x for _, x in out]))
    print(f"paper frames: {time.time() - t0:.0f}s")
frames = np.load(CACHE["frames"])
paper_rot, frame_l1 = frames["rot"], frames["l1"]
print(f"published ECT reproduced exactly for {np.sum(frame_l1 == 0)} of {len(frame_l1)} seeds; "
      f"fixed sign rule for seeds {np.flatnonzero(frame_l1 > 0).tolist()} (no sign choice comes close)")

# recompute a sample on this machine: the matched frame reproduces the published ECT and the cache
exact, flipped = [], 0
for i in check:
    coords, cells = B.cubical_complex(B.load_seed(files[i]))
    center, rot, dist = B.matched_paper_frame(coords, cells, published_ect64[i])
    exact.append(dist == 0)
    flipped += not np.allclose(rot, B.paper_frame(coords)[1])
    assert np.allclose(rot, paper_rot[i], atol=1e-8), i
print(f"published ECT reproduced exactly for {sum(exact)} of {len(exact)} randomly chosen seeds "
      f"({flipped} of them needed a sign flip of this machine's SVD axes)")
assert all(exact)


## Per-seed processing

In [ ]:
SWAP = np.array([[1, 0, 0], [0, 0, 1], [0, 1, 0]], float)


def process_seed(i, path, rot):
    """DIWECT of the mesh, and pyECT's WECT of the mesh in every pose of POSES (`rot`: the paper's frame)."""
    torch.set_num_threads(1)
    img = B.load_seed(path)
    center = np.argwhere(img).astype(float).mean(0)
    verts, faces = B.mesh(img, center)
    rng = np.random.default_rng([B.SEED, i])
    random_rot = B.random_rotation(rng)
    axis = rng.normal(size=3)
    axis /= np.linalg.norm(axis)
    s_center, s_axes, variances = B.surface_frame(verts, faces)
    in_paper = verts @ rot
    posed = {"paper": in_paper, "automatic PCA": (verts - s_center) @ s_axes,
             "scanner": verts, "random rotation": verts @ random_rot.T}
    for t in PERTURBATIONS:
        posed[f"paper + {t}°"] = in_paper @ B.axis_rotation(axis, t).T
    cx = B.simplicial_complex(verts, faces)
    wect = WECT(torch.tensor(B.DIRECTIONS, dtype=torch.float32), B.NUM_THRESHOLDS)
    with torch.no_grad():
        diwect = DIWECT(NUM_HEIGHTS_DIWECT, normalize=True)(cx).numpy()
        wects = np.stack([wect([(torch.tensor(posed[p]), cx[0][1]), cx[1], cx[2]]).flatten().numpy()
                          for p in POSES]).astype(np.float32)
    chi = len(verts) - len(cx[1][0]) + len(faces)
    return dict(diwect=diwect, wect=wects, chi=chi, triangles=len(faces), variances=variances,
                paper_rot=rot, surface_axes=s_axes)


def process_flip(i, path, rot):
    """Global-grid WECT of the mesh in the paper's frame with simulated PCA sign flips / axis swaps."""
    img = B.load_seed(path)
    center = np.argwhere(img).astype(float).mean(0)
    verts, faces = B.mesh(img, center)
    edges = B.mesh_edges(faces)
    rng = np.random.default_rng([B.SEED, 7, i])
    signs = np.diag(rng.choice([-1.0, 1.0], 3))
    swap = SWAP if rng.random() < 0.5 else np.eye(3)
    p = verts @ rot
    posed = {"paper": p, "sign flips": p @ signs, "width/thickness swap": p @ swap, "swap + sign flips": p @ swap @ signs}
    return np.stack([B.ect_both_binnings(posed[k], [edges, faces], B.DIRECTIONS, B.NUM_THRESHOLDS)[1]
                     for k in FLIP_POSES]).astype(np.int32)


def process_binning(i, path, rot):
    """WECT of the voxel complex and the mesh, min-max and global binning, paper's frame and random rotation."""
    img = B.load_seed(path)
    coords, cells = B.cubical_complex(img)
    center = coords.mean(0)
    random_rot = B.random_rotation(np.random.default_rng([B.SEED, i]))   # the same rotation as process_seed
    verts, faces = B.mesh(img, center)
    complexes = {"voxel": (coords - center, cells), "mesh": (verts, [B.mesh_edges(faces), faces])}
    out = {}
    for cx, (v, c) in complexes.items():
        for p, posed in [("paper", v @ rot), ("random rotation", v @ random_rot.T)]:
            mm, gl = B.ect_both_binnings(posed, c, B.DIRECTIONS, B.NUM_THRESHOLDS)
            out[cx, "min-max", p], out[cx, "global", p] = mm, gl
    return np.stack([out[k] for k in BIN_VARIANTS]).astype(np.int32)


def run_all(fn, indices):
    return Parallel(n_jobs=N_JOBS)(delayed(fn)(i, files[i], paper_rot[i]) for i in indices)

## Compute or load the caches

In [ ]:
everything = range(len(files))

if RECOMPUTE or not CACHE["features"].exists():
    t0 = time.time()
    out = run_all(process_seed, everything)
    np.savez_compressed(CACHE["features"], **{k: np.stack([o[k] for o in out]) for k in out[0]})
    print(f"features: {time.time() - t0:.0f}s")
if RECOMPUTE or not CACHE["flipflop"].exists():
    t0 = time.time()
    np.savez_compressed(CACHE["flipflop"], ect=np.stack(run_all(process_flip, everything)))
    print(f"flip-flop poses: {time.time() - t0:.0f}s")
if RECOMPUTE or not CACHE["binning"].exists():
    t0 = time.time()
    np.savez_compressed(CACHE["binning"], ect=np.stack(run_all(process_binning, everything)))
    print(f"binning: {time.time() - t0:.0f}s")

feats = dict(np.load(CACHE["features"]))
flipped = np.load(CACHE["flipflop"])["ect"]
binned = np.load(CACHE["binning"])["ect"]

# caches computed with a different paper frame (e.g., before the frames were matched to the
# published ECT): recompute those seeds
stale = np.flatnonzero(np.abs(feats["paper_rot"] - paper_rot).max((1, 2)) > 1e-8)
if len(stale):
    print(f"recomputing {len(stale)} seeds whose cached paper frame differs: {stale.tolist()}")
    for i, o in zip(stale, run_all(process_seed, stale)):
        for k in feats:
            feats[k][i] = o[k]
    flipped[stale] = np.stack(run_all(process_flip, stale))
    binned[stale] = np.stack(run_all(process_binning, stale))
    np.savez_compressed(CACHE["features"], **feats)
    np.savez_compressed(CACHE["flipflop"], ect=flipped)
    np.savez_compressed(CACHE["binning"], ect=binned)
print({k: v.shape for k, v in feats.items()}, flipped.shape, binned.shape)

## Verify the caches on a sample of seeds

Recomputes `VERIFY` seeds with the code above and compares with the cached values.

In [ ]:
if VERIFY:
    sample = check[:VERIFY]
    new = run_all(process_seed, sample)
    for k in ["diwect", "wect", "chi", "triangles", "variances", "paper_rot", "surface_axes"]:
        got = np.stack([o[k] for o in new])
        err = np.abs(got - feats[k][sample]).max()
        print(f"features[{k!r}]: max abs difference {err:.2e}")
        assert np.allclose(got, feats[k][sample], rtol=1e-6, atol=1e-6), k
    assert np.array_equal(np.stack(run_all(process_flip, sample)), flipped[sample])
    assert np.array_equal(np.stack(run_all(process_binning, sample)), binned[sample])
    print(f"flip-flop and binning caches: identical on {len(sample)} seeds")

## Consistency checks and summary

In [ ]:
bins = {k: binned[:, j] for j, k in enumerate(BIN_VARIANTS)}
print("voxel complex, min-max binning, paper's frame == published ECT (T=16) for "
      f"{np.mean((bins['voxel', 'min-max', 'paper'] == d['published_ect']).all(1)):.1%} of seeds")
for p, k in [("paper", 0), ("random rotation", 3)]:
    same = np.abs(bins["mesh", "global", p] - feats["wect"][:, k]).max(1) == 0
    print(f"mesh, global grid, {p} == pyECT's WECT for {same.mean():.1%} of seeds")
print("flip-flop 'paper' pose == mesh, global grid, paper for "
      f"{np.mean((flipped[:, 0] == bins['mesh', 'global', 'paper']).all(1)):.1%} of seeds")

# The DIWECT does not depend on the orientation: recompute it after random rotations.
inv = []
for i in check[:4]:
    img = B.load_seed(files[i])
    v, f = B.mesh(img, np.argwhere(img).mean(0))
    for k in range(2):
        rot = B.random_rotation(np.random.default_rng([B.SEED, 99, i, k]))
        with torch.no_grad():
            dw = DIWECT(NUM_HEIGHTS_DIWECT, normalize=True)(B.simplicial_complex(v @ rot.T, f)).numpy()
        inv.append(np.abs(dw - feats["diwect"][i]).max() / np.abs(feats["diwect"][i]).max())
print(f"DIWECT after random rotations: max relative difference {max(inv):.1e}")

print(f"triangles per seed: median {int(np.median(feats['triangles']))} "
      f"(range {feats['triangles'].min()}-{feats['triangles'].max()}); Euler characteristic: "
      f"{dict(zip(*[x.tolist() for x in np.unique(feats['chi'], return_counts=True)]))}")